# Store Sales Forecasting - End-to-End Pipeline

This notebook executes data cleaning, feature engineering, saves processed datasets to `data/processed/`, and generates all 9 exploratory plots saved to the `visualization/` directory.

In [ ]:
import sys
import os
sys.path.append("..")

from src.preprocessing.load_data import load_raw_data
from src.preprocessing.holidays import process_holidays
from src.preprocessing.oil import process_oil
from src.preprocessing.transactions import process_transactions
from src.preprocessing.clean_data import filter_pre_opening_days
from src.features.build_features import build_all_features
from src.visualization.plots import plot_all_visualizations

## 1. Data Cleaning & Preprocessing

In [ ]:
# Load raw datasets
df, stores_df, transactions_df, oil_df, holidays_df, test_df = load_raw_data("../data/raw")
print("Raw merged df shape:", df.shape)

# 1. Process Holidays
df = process_holidays(df, holidays_df)
test_df = process_holidays(test_df, holidays_df)

# 2. Process Oil Prices
df, full_oil_df = process_oil(df, oil_df)
test_df, _ = process_oil(test_df, oil_df)

# 3. Process Transactions (Linear regression imputation & closed days)
df, daily_sales = process_transactions(df, transactions_df)

# 4. Filter Pre-Opening Zero-Sales Days
df = filter_pre_opening_days(df)
print("Cleaned df shape:", df.shape)

## 2. Feature Engineering & Saving Processed Output

In [ ]:
# Build all manufactured features
df = build_all_features(df)
print("Final Feature Set Shape:", df.shape)

# Save processed datasets to data/processed/
os.makedirs("../data/processed", exist_ok=True)
processed_path = "../data/processed/train_processed.parquet"
df.to_parquet(processed_path, index=False)
print(f"Processed training dataset saved to: {processed_path}")

## 3. Exploratory Data Visualizations (Saved to visualization/)

In [ ]:
# Generate, display, and save all 9 plots to ../visualization/
plot_all_visualizations(df, full_oil_df, output_dir="../visualization")